In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Geometry-V7 R3 exploratory cycle diagnostic
This thin notebook binds a reviewed producer exact and invokes the create-only runner once.

In [ ]:
import re
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
APPROVED_EXACT = 'PENDING_AFTER_GEOMETRY_V7_R3_PUSH'
if re.fullmatch(r'[0-9a-f]{40}', APPROVED_EXACT) is None:
    raise RuntimeError('Bind the approved pushed Geometry-V7 R3 exact before execution')
checkout = Path('/content/CEG-WM')
if checkout.exists():
    raise FileExistsError(f'create-only checkout already exists: {checkout}')
subprocess.run(['git', 'clone', REPO_URL, str(checkout)], check=True)
subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', APPROVED_EXACT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(checkout)], check=True)

In [ ]:
import torch

assert torch.cuda.is_available(), 'GPU required for official SyncSeal D4 probes'
R1A_ARTIFACT_ROOT = Path('/content/drive/MyDrive/CEG-WM/Geometry-V7/ac590330e91aacf4b3283df1e94572a0e4f983a0/r1a-f2')
R1B_REPAIR_ARTIFACT_ROOT = Path('/content/drive/MyDrive/CEG-WM/Geometry-V7/3b9819d80b07704a4caab8b7aaa581cf9eb8a3c5/r1b-repair')
R2_ARTIFACT_ROOT = Path('/content/drive/MyDrive/CEG-WM/Geometry-V7/ffac9d4c1e575c27240d9423bbd30e0713aa2dcd/r2-selective')
LOCAL_RESULT_DIR = Path('/content/geometry_v7_r3_exploratory_result')
SYNCSEAL_CHECKPOINT = Path('/content/checkpoints/r3_syncmodel.jit.pt')
DRIVE_RESULT_DIR = Path('/content/drive/MyDrive/CEG-WM/Geometry-V7') / APPROVED_EXACT / 'r3-exploratory'
if not all(path.is_dir() for path in (R1A_ARTIFACT_ROOT, R1B_REPAIR_ARTIFACT_ROOT, R2_ARTIFACT_ROOT)):
    raise FileNotFoundError('fixed accepted R1A, R1B-repair, or R2 artifact is absent')
if LOCAL_RESULT_DIR.exists() or SYNCSEAL_CHECKPOINT.exists() or DRIVE_RESULT_DIR.exists():
    raise FileExistsError('create-only R3 destination already exists')
command = [
    sys.executable, '-m', 'experiments.run_geometry_v7_r3',
    '--repo-root', str(checkout), '--expected-exact', APPROVED_EXACT,
    '--r1a-artifact-root', str(R1A_ARTIFACT_ROOT),
    '--r1b-repair-artifact-root', str(R1B_REPAIR_ARTIFACT_ROOT),
    '--r2-artifact-root', str(R2_ARTIFACT_ROOT),
    '--syncseal-checkpoint', str(SYNCSEAL_CHECKPOINT),
    '--result-dir', str(LOCAL_RESULT_DIR),
]
completed = subprocess.run(command, cwd=checkout, text=True, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, check=False)
print(completed.stdout.strip())
if completed.returncode not in (0, 2) or not (LOCAL_RESULT_DIR / 'result.json').is_file():
    raise RuntimeError('Geometry-V7 R3 runner stopped before a complete package')

In [ ]:
import shutil

if not (LOCAL_RESULT_DIR / 'result.json').is_file():
    raise RuntimeError('Run the bound R3 producer before publication')
if DRIVE_RESULT_DIR.exists():
    raise FileExistsError(f'create-only Drive result already exists: {DRIVE_RESULT_DIR}')
DRIVE_RESULT_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(LOCAL_RESULT_DIR, DRIVE_RESULT_DIR)
print(DRIVE_RESULT_DIR)